In [1]:
%pip install -q gradio
print("Done.")

Note: you may need to restart the kernel to use updated packages.
Done.


In [1]:
import os, json, re, time, requests, gradio as gr
from pathlib import Path
from datetime import datetime
from typing import Optional, TypedDict
from tenacity import retry, stop_after_attempt, wait_exponential
from sqlalchemy import create_engine, Column, String, Integer, Float, Text
from sqlalchemy.orm import declarative_base, Session
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, END
import chromadb

#Setup
ROOT = Path(r"C:\Users\MAITHILI\Care_Companion")

os.environ["GROQ_API_KEY"] = "**************"
os.environ["USDA_API_KEY"] = "*************"

print(f"ROOT: {ROOT}")
print(f"DB exists: {(ROOT / 'memory' / 'sqlite' / 'carecompanion.db').exists()}")

#Database
Base    = declarative_base()
DB_PATH = ROOT / "memory" / "sqlite" / "carecompanion.db"
engine  = create_engine(f"sqlite:///{DB_PATH}", echo=False)

class Patient(Base):
    __tablename__    = "patients"
    patient_id       = Column(String, primary_key=True)
    name             = Column(String)
    age              = Column(Integer)
    created_at       = Column(String)
    last_seen        = Column(String)
    conditions       = Column(Text)
    medications      = Column(Text)
    allergies        = Column(Text)
    recent_glucose   = Column(Float)
    recent_a1c       = Column(Float)
    recent_bmi       = Column(Float)
    diet_preferences = Column(Text)
    foods_to_avoid   = Column(Text)
    activity_level   = Column(String)
    budget_monthly   = Column(Float)
    insurance        = Column(String)
    session_count    = Column(Integer, default=0)
    agent_notes      = Column(Text)
    last_fda_check   = Column(String, nullable=True)

Base.metadata.create_all(engine)

def load_patient(pid: str) -> Optional[dict]:
    with Session(engine) as s:
        r = s.get(Patient, pid)
        if not r: return None
        return {
            "patient_id":       r.patient_id,
            "name":             r.name,
            "age":              r.age,
            "created_at":       r.created_at,
            "last_seen":        r.last_seen,
            "conditions":       json.loads(r.conditions),
            "medications":      json.loads(r.medications),
            "allergies":        json.loads(r.allergies),
            "recent_glucose":   r.recent_glucose,
            "recent_a1c":       r.recent_a1c,
            "recent_bmi":       r.recent_bmi,
            "diet_preferences": json.loads(r.diet_preferences),
            "foods_to_avoid":   json.loads(r.foods_to_avoid),
            "activity_level":   r.activity_level,
            "budget_monthly":   r.budget_monthly,
            "insurance":        r.insurance,
            "session_count":    r.session_count,
            "agent_notes":      json.loads(r.agent_notes),
            "last_fda_check":   r.last_fda_check,
        }

def update_field(pid, field, value):
    with Session(engine) as s:
        r = s.get(Patient, pid)
        if r:
            setattr(r, field,
                    json.dumps(value) if isinstance(value, (list, dict))
                    else value)
            r.last_seen = datetime.now().isoformat()
            s.commit()

def all_patients():
    with Session(engine) as s:
        return [
            {"patient_id": r.patient_id, "name": r.name,
             "age": r.age, "session_count": r.session_count}
            for r in s.query(Patient).all()
        ]

#ChromaDB 
chroma_client = chromadb.PersistentClient(
    path=str(ROOT / "memory" / "chroma_db"))
col = chroma_client.get_or_create_collection(
    "conversations", metadata={"hnsw:space": "cosine"})

def save_msg(pid, role, content, sid):
    col.add(
        documents=[content],
        metadatas=[{"patient_id": pid, "role": role,
                    "session_id": sid,
                    "timestamp":  datetime.now().isoformat()}],
        ids=[f"{pid}_{sid}_{datetime.now().timestamp()}"]
    )

def recall(pid, query, n=3):
    count = col.count()
    if count == 0: return []
    res = col.query(
        query_texts=[query],
        n_results=min(n, count),
        where={"patient_id": pid}
    )
    if not res["documents"][0]: return []
    return [{"role": m["role"], "content": d}
            for d, m in zip(res["documents"][0], res["metadatas"][0])]

#Tools
@retry(stop=stop_after_attempt(3), wait=wait_exponential(min=1, max=5))
def drug_interactions(meds):
    out = {}
    for med in meds:
        name = med["name"] if isinstance(med, dict) else med
        try:
            r = requests.get(
                "https://api.fda.gov/drug/label.json",
                params={"search": f'openfda.generic_name:"{name.lower()}"',
                        "limit": 1}, timeout=10)
            if r.status_code != 200:
                out[name] = {"status": "not_found"}; continue
            lb = r.json()["results"][0]
            out[name] = {
                "status":       "found",
                "warnings":     lb.get("warnings",     ["None"])[0][:200],
                "interactions": lb.get("drug_interactions", ["None"])[0][:300],
            }
        except Exception as e:
            out[name] = {"status": "error", "error": str(e)}
    return out

def med_cost(name):
    P = {
        "metformin":        {"brand": "Glucophage", "bp": 45,  "gp": 4},
        "glipizide":        {"brand": "Glucotrol",  "bp": 38,  "gp": 9},
        "sitagliptin":      {"brand": "Januvia",    "bp": 550, "gp": 45},
        "lisinopril":       {"brand": "Zestril",    "bp": 65,  "gp": 8},
        "insulin glargine": {"brand": "Lantus",     "bp": 340, "gp": 98},
    }
    i = P.get(name.lower())
    if i:
        s = i["bp"] - i["gp"]
        return {"generic": f"${i['gp']}/mo", "savings": f"${s*12}/yr",
                "tip": f"Ask for generic {name} instead of {i['brand']}. Save ${s}/month."}
    return {"tip": f"Ask your pharmacist about generics for {name}."}

def nutrition(food):
    try:
        r = requests.get(
            "https://api.nal.usda.gov/fdc/v1/foods/search",
            params={"query": food, "pageSize": 1,
                    "api_key": os.environ.get("USDA_API_KEY", "DEMO_KEY")},
            timeout=15)
        foods = r.json().get("foods", [])
        if not foods: return {"status": "not_found"}
        n  = {x["nutrientName"]: x["value"]
              for x in foods[0].get("foodNutrients", [])}
        nc = max(0, n.get("Carbohydrate, by difference", 0)
                  - n.get("Fiber, total dietary", 0))
        flag   = "green" if nc<5 else "yellow" if nc<15 else "orange" if nc<30 else "red"
        labels = {"green":  "Excellent for blood sugar",
                  "yellow": "Moderate — control portions",
                  "orange": "Caution — limit portions",
                  "red":    "Avoid — spikes blood sugar"}
        return {"net_carbs": round(nc, 1), "flag": flag,
                "label": labels[flag], "status": "found"}
    except:
        return {"status": "error"}

def fda_alerts(meds):
    out = {}
    for med in meds:
        name = med["name"] if isinstance(med, dict) else med
        hits = []
        try:
            r = requests.get(
                "https://api.fda.gov/drug/enforcement.json",
                params={"search": f'product_description:"{name}"',
                        "limit": 2}, timeout=10)
            if r.status_code == 200:
                hits = [x.get("reason_for_recall", "")[:100]
                        for x in r.json().get("results", [])]
        except: pass
        out[name] = {"count": len(hits), "alerts": hits,
                     "summary": "No active alerts." if not hits
                                else f"{len(hits)} alert(s) found."}
    return out

def pubmed(query, n=2):
    try:
        ids = requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
            params={"db": "pubmed", "term": query,
                    "retmax": n, "retmode": "json"},
            timeout=10).json()["esearchresult"]["idlist"]
        if not ids: return []
        sm = requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi",
            params={"db": "pubmed", "id": ",".join(ids), "retmode": "json"},
            timeout=10).json().get("result", {})
        return [{"title": sm.get(p, {}).get("title", ""),
                 "year":  sm.get(p, {}).get("pubdate", "")[:4],
                 "url":   f"https://pubmed.ncbi.nlm.nih.gov/{p}/"}
                for p in ids]
    except: return []

#LangGraph Agent
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2,
               api_key=os.environ["GROQ_API_KEY"])

class State(TypedDict):
    pid: str; msg: str; profile: dict; history: list
    tools: list; results: dict; response: str; sid: str

def node_context(state):
    p = load_patient(state["pid"])
    p["session_count"] += 1
    update_field(state["pid"], "session_count", p["session_count"])
    sid = f"s_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    return {**state, "profile": p,
            "history": recall(state["pid"], state["msg"], 4),
            "sid": sid}

def node_tools(state):
    p = state["profile"]
    r = llm.invoke([HumanMessage(content=
        f"Which tools are needed for this patient query?\n"
        f"Medications: {[m['name'] for m in p['medications']]}\n"
        f"Message: '{state['msg']}'\n"
        f"Available: check_interactions, med_cost, nutrition, fda_alerts, pubmed\n"
        f"Reply ONLY with a JSON list like ['nutrition'] or []"
    )])
    try:
        match = re.search(r'\[.*?\]', r.content, re.DOTALL)
        tools = json.loads(match.group()) if match else []
    except: tools = []
    if p["session_count"] <= 1 and "fda_alerts" not in tools:
        tools.append("fda_alerts")
    return {**state, "tools": tools}

def node_run(state):
    meds = state["profile"]["medications"]
    out  = {}
    food_kw = ["rice","bread","oatmeal","pasta","potato","fruit","sugar",
               "juice","milk","chicken","fish","egg","cheese","wheat",
               "carb","vegetable","meat","cereal","bean"]
    for tool in state["tools"]:
        try:
            if tool == "check_interactions":
                out["interactions"] = drug_interactions(meds)
            elif tool == "med_cost":
                out["costs"] = {m["name"]: med_cost(m["name"]) for m in meds}
            elif tool == "nutrition":
                foods = [f for f in food_kw
                         if f in state["msg"].lower()][:2]
                if not foods: foods = ["the food mentioned"]
                out["nutrition"] = {f: nutrition(f) for f in foods}
            elif tool == "fda_alerts":
                out["fda_alerts"] = fda_alerts(meds)
            elif tool == "pubmed":
                out["research"] = pubmed(
                    state["msg"] + " type 2 diabetes", 2)
        except Exception as e:
            out[tool] = {"error": str(e)}
    return {**state, "results": out}

def node_respond(state):
    p    = state["profile"]
    meds = ", ".join(f"{m['name']} {m['dose']}" for m in p["medications"])
    hist = "\n".join(
        f"[{m['role'].upper()}]: {m['content'][:80]}"
        for m in state["history"]) or "First session."
    notes = "; ".join(p["agent_notes"][-2:]) if p["agent_notes"] else "none"

    prompt = f"""You are CareCompanion AI for Type 2 Diabetes patients.
Be warm, personalized, and clear. End with one follow-up question.
Always remind the patient you are not a replacement for their doctor.

PATIENT:
- Name: {p['name']}, Age: {p['age']}
- Conditions: {', '.join(p['conditions'])}
- Medications: {meds}
- Glucose: {p['recent_glucose']} mg/dL | A1c: {p['recent_a1c']}% | BMI: {p['recent_bmi']}
- Diet: {', '.join(p['diet_preferences'])} | Budget: ${p['budget_monthly']}/mo
- Insurance: {p['insurance']} | Sessions: {p['session_count']}
- Agent notes: {notes}

PAST CONVERSATIONS:
{hist}

TOOL DATA:
{json.dumps(state['results'], indent=1)[:1200]}

PATIENT SAYS: "{state['msg']}"

Write a warm, helpful, personalized response."""

    resp = llm.invoke([HumanMessage(content=prompt)])

    #saving to memory
    save_msg(state["pid"], "user",      state["msg"],    state["sid"])
    save_msg(state["pid"], "assistant", resp.content,    state["sid"])


    note_resp = llm.invoke([HumanMessage(content=
        f"One sentence: key thing to remember. "
        f"Patient said: '{state['msg']}'. Only the sentence.")])
    notes_list = p["agent_notes"] + \
        [f"[{datetime.now().strftime('%Y-%m-%d')}] {note_resp.content.strip()}"]
    update_field(state["pid"], "agent_notes", notes_list[-10:])

    return {**state, "response": resp.content}

#building graph
graph = StateGraph(State)
graph.add_node("ctx",     node_context)
graph.add_node("tools",   node_tools)
graph.add_node("run",     node_run)
graph.add_node("respond", node_respond)
graph.set_entry_point("ctx")
graph.add_edge("ctx",     "tools")
graph.add_edge("tools",   "run")
graph.add_edge("run",     "respond")
graph.add_edge("respond", END)
agent = graph.compile()

def run_agent(pid, msg):
    result = agent.invoke({
        "pid": pid, "msg": msg, "profile": {},
        "history": [], "tools": [], "results": {},
        "response": "", "sid": ""
    })
    return result["response"], result["tools"]

print("Agent ready.")
patients = all_patients()
print(f"Patients loaded: {len(patients)}")

ROOT: C:\Users\MAITHILI\Care_Companion
DB exists: True
Agent ready.
Patients loaded: 20


In [2]:
#patient dropdown options
patient_map = {
    f"{p['name']} (ID: {p['patient_id']})": p["patient_id"]
    for p in all_patients()
}

def get_profile_text(patient_label):
    """Returns patient profile as formatted text for the sidebar."""
    pid     = patient_map[patient_label]
    profile = load_patient(pid)
    if not profile:
        return "Patient not found."

    meds = "\n".join(
        f"  💊 {m['name']} {m['dose']} — {m['frequency']}"
        for m in profile["medications"]
    )

    alerts     = fda_alerts(profile["medications"])
    alert_text = ""
    for drug, info in alerts.items():
        if info["count"] > 0:
            alert_text += f"\n⚠️ FDA Alert — {drug}: {info['summary']}"

    notes = "\n".join(
        f"  📝 {n}" for n in profile["agent_notes"][-3:]
    ) if profile["agent_notes"] else "  None yet."

    return f"""
╔══════════════════════════════╗
   {profile['name']}'s Profile
╚══════════════════════════════╝

📊 VITALS
  Age:     {profile['age']}
  BMI:     {profile['recent_bmi']}
  Glucose: {profile['recent_glucose']} mg/dL
  A1c:     {profile['recent_a1c']}%

💊 MEDICATIONS
{meds}

🏥 INSURANCE & BUDGET
  Insurance: {profile['insurance']}
  Budget:    ${profile['budget_monthly']}/month

🍽️ DIET & LIFESTYLE
  Diet:     {', '.join(profile['diet_preferences'])}
  Activity: {profile['activity_level']}

💬 SESSIONS: {profile['session_count']}
{alert_text}

📋 AGENT NOTES
{notes}
""".strip()


def respond(patient_label, message, chat_history):
    """Main chat function called by Gradio."""
    if not patient_label:
        return chat_history + [["", "Please select a patient first."]]
    if not message.strip():
        return chat_history

    pid = patient_map[patient_label]

    #running the agent
    response, tools_used = run_agent(pid, message)

    #adding tools info to response
    if tools_used:
        tools_str = ", ".join(f"`{t}`" for t in tools_used)
        response += f"\n\n🔧 *Tools used: {tools_str}*"

    chat_history.append([message, response])
    return chat_history


def clear_chat():
    return []


def update_profile(patient_label):
    if not patient_label:
        return "Select a patient to see their profile."
    return get_profile_text(patient_label)


#building Gradio Interface 
with gr.Blocks(
    title="CareCompanion AI",
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="teal",
    )
) as demo:

    # Header
    gr.Markdown("""
    # 🩺 CareCompanion AI
    ### Persistent Memory Agent for Type 2 Diabetes Management
    *Remembers you across every session · Powered by LangGraph + Groq*
    ---
    """)

    with gr.Row():

        # Left column - patient selector + profile
        with gr.Column(scale=1):
            gr.Markdown("### Select Patient")
            patient_dropdown = gr.Dropdown(
                choices=list(patient_map.keys()),
                value=list(patient_map.keys())[0],
                label="Patient",
                interactive=True
            )

            gr.Markdown("### Patient Profile")
            profile_box = gr.Textbox(
                value=get_profile_text(list(patient_map.keys())[0]),
                label="",
                lines=28,
                interactive=False,
                show_label=False
            )

        # Right column - chat
        with gr.Column(scale=2):
            gr.Markdown("### Chat")
            chatbot = gr.Chatbot(
                label="",
                height=480,
                bubble_full_width=False,
                avatar_images=(
                    "https://img.icons8.com/color/48/user-male-circle.png",
                    "https://img.icons8.com/color/48/caduceus.png"
                )
            )

            with gr.Row():
                msg_box = gr.Textbox(
                    placeholder="Ask me about your medications, diet, costs...",
                    label="",
                    scale=4,
                    lines=1
                )
                send_btn = gr.Button("Send 💬", scale=1, variant="primary")

            clear_btn = gr.Button("Clear Chat 🗑️", variant="secondary")

            gr.Markdown("""
            **Try asking:**
            - *"Is oatmeal okay for breakfast?"*
            - *"How much could I save on my medications?"*
            - *"My glucose was 132 today, is that progress?"*
            - *"What foods should I avoid with Metformin?"*
            """)

    #Event handlers
    patient_dropdown.change(
        fn=update_profile,
        inputs=[patient_dropdown],
        outputs=[profile_box]
    )

    send_btn.click(
        fn=respond,
        inputs=[patient_dropdown, msg_box, chatbot],
        outputs=[chatbot]
    ).then(
        fn=lambda: "",
        outputs=[msg_box]
    )

    msg_box.submit(
        fn=respond,
        inputs=[patient_dropdown, msg_box, chatbot],
        outputs=[chatbot]
    ).then(
        fn=lambda: "",
        outputs=[msg_box]
    )

    clear_btn.click(
        fn=clear_chat,
        outputs=[chatbot]
    )

#Launch
demo.launch(
    share=True,        # gives you a public URL instantly
    inbrowser=True,    # opens browser automatically
    server_port=7860,
)

Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
